## Key Takeaways

1. **OLS is biased** when treatment is endogenous (correlated with unobserved confounders).

2. **An instrument** is a variable that shifts treatment but has no direct effect on the outcome. Three conditions: relevance, exclusion, independence.

3. **2SLS** isolates the "clean" variation in treatment (coming from the instrument) and uses only that to estimate the causal effect.

4. **Weak instruments** (F < 10) make IV estimates noisy and biased toward OLS. Always report the first-stage F.

5. **LATE**: IV estimates the effect for compliers only, not for everyone. If treatment effects vary across people, IV and ATE can differ.

6. **The exclusion restriction is untestable.** You can only argue for it based on your research design. A small violation can meaningfully bias the IV estimate.

**Real-world examples from this week's readings:**
- Djourelova (2023): AP style ban as instrument for slanted language exposure
- Mueller & Schwarz (2021): internet outages as instrument for social media exposure

In [ ]:
# What if Z has a direct effect on Y?
# Y = 2 + 3*D + delta*Z + 1.5*U + noise
# When delta != 0, the exclusion restriction is violated

np.random.seed(42)
deltas = np.arange(-1.0, 1.01, 0.05)
iv_estimates = []

for delta in deltas:
    U_ex = np.random.normal(0, 1, n)
    Z_ex = np.random.normal(0, 1, n)
    D_ex = 0.6 * Z_ex + 0.5 * U_ex + np.random.normal(0, 1, n)
    Y_ex = 2 + true_effect * D_ex + delta * Z_ex + 1.5 * U_ex + np.random.normal(0, 1, n)

    cov_yz = np.cov(Y_ex, Z_ex)[0, 1]
    cov_dz = np.cov(D_ex, Z_ex)[0, 1]
    iv_estimates.append(cov_yz / cov_dz)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(deltas, iv_estimates, 'o-', color='steelblue', markersize=4)
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5, label='Exclusion holds (delta = 0)')
ax.set_xlabel('Direct effect of Z on Y (delta)', fontsize=12)
ax.set_ylabel('IV estimate', fontsize=12)
ax.set_title('IV Bias When the Exclusion Restriction is Violated', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print(f"When delta = 0, IV gives {true_effect:.1f} (correct).")
print(f"When delta > 0, Z directly helps Y, making treatment look more effective.")
print(f"When delta < 0, Z directly hurts Y, making treatment look less effective.")
print(f"\nThe exclusion restriction cannot be tested statistically.")
print(f"You can only argue for it based on the research design.")

## 6. Exclusion Restriction Violation

The exclusion restriction says $Z$ affects $Y$ *only* through $D$. If $Z$ has even a small direct effect on $Y$, the IV estimate is biased. Unlike relevance (which we can test with F-statistics), the exclusion restriction is **untestable**. We can only argue for it on theoretical grounds.

In [ ]:
# Simulate compliers, always-takers, never-takers with DIFFERENT treatment effects
np.random.seed(42)
n_late = 6000

# Assign types: 50% compliers, 25% always-takers, 25% never-takers
types = np.random.choice(['complier', 'always-taker', 'never-taker'],
                         size=n_late, p=[0.5, 0.25, 0.25])

# Binary instrument
Z_late = np.random.binomial(1, 0.5, n_late)

# Treatment depends on type and instrument
D_late = np.zeros(n_late)
D_late[types == 'always-taker'] = 1  # always treated
D_late[types == 'never-taker'] = 0   # never treated
D_late[(types == 'complier') & (Z_late == 1)] = 1  # treated when Z=1
D_late[(types == 'complier') & (Z_late == 0)] = 0  # untreated when Z=0

# Different treatment effects by type
tau = np.zeros(n_late)
tau[types == 'complier'] = 3.0       # complier effect
tau[types == 'always-taker'] = 1.0   # always-taker effect (smaller)
tau[types == 'never-taker'] = 5.0    # never-taker effect (larger, but unobserved)

Y_late = 2 + tau * D_late + np.random.normal(0, 1, n_late)

# Compute IV (Wald) estimate
mean_Y_Z1 = Y_late[Z_late == 1].mean()
mean_Y_Z0 = Y_late[Z_late == 0].mean()
mean_D_Z1 = D_late[Z_late == 1].mean()
mean_D_Z0 = D_late[Z_late == 0].mean()
iv_late = (mean_Y_Z1 - mean_Y_Z0) / (mean_D_Z1 - mean_D_Z0)

# True effects
ate = (tau * D_late).sum() / D_late.sum() if D_late.sum() > 0 else 0
complier_ate = tau[types == 'complier'].mean()

print("Treatment effects by type:")
print(f"  Compliers:      tau = 3.0")
print(f"  Always-takers:  tau = 1.0")
print(f"  Never-takers:   tau = 5.0 (never observed getting treated)")
print()
print(f"IV/Wald estimate: {iv_late:.3f}")
print(f"True complier effect: {complier_ate:.1f}")
print(f"\nIV recovers the complier effect, NOT the population ATE.")
print(f"This is LATE: the Local Average Treatment Effect.")

# Visualize the three groups
fig, ax = plt.subplots(figsize=(10, 5))
for i, (t, color) in enumerate([('complier', 'steelblue'),
                                  ('always-taker', 'darkorange'),
                                  ('never-taker', 'gray')]):
    mask = types == t
    counts = [np.sum(mask & (Z_late == 0) & (D_late == 0)),
              np.sum(mask & (Z_late == 0) & (D_late == 1)),
              np.sum(mask & (Z_late == 1) & (D_late == 0)),
              np.sum(mask & (Z_late == 1) & (D_late == 1))]
    ax.bar([x + i*0.25 for x in range(4)], counts, width=0.25,
           color=color, label=t.replace('-', ' ').title(), edgecolor='black')

ax.set_xticks([0.25, 1.25, 2.25, 3.25])
ax.set_xticklabels(['Z=0, D=0', 'Z=0, D=1', 'Z=1, D=0', 'Z=1, D=1'])
ax.set_ylabel('Count')
ax.set_title('Who does what? Compliers, Always-takers, Never-takers')
ax.legend()
plt.tight_layout()
plt.show()

print("\nCompliers (blue) switch treatment status when Z changes.")
print("Always-takers (orange) are always treated regardless of Z.")
print("Never-takers (gray) are never treated regardless of Z.")

## 5. LATE: Who Does IV Apply To?

IV does not estimate the average treatment effect (ATE) for the whole population. It estimates the **Local Average Treatment Effect (LATE)**: the effect for **compliers**, people whose treatment status is changed by the instrument.

Three types of people:
- **Compliers**: take treatment when $Z=1$, don't when $Z=0$ (IV estimates their effect)
- **Always-takers**: take treatment regardless of $Z$ (IV tells us nothing about them)
- **Never-takers**: never take treatment regardless of $Z$ (IV tells us nothing about them)

In [ ]:
# Vary instrument strength and track IV estimate + F-statistic
strengths = np.arange(0.02, 1.01, 0.02)
results = []

for gamma in strengths:
    np.random.seed(42)
    U_s = np.random.normal(0, 1, n)
    Z_s = np.random.normal(0, 1, n)
    D_s = gamma * Z_s + 0.5 * U_s + np.random.normal(0, 1, n)
    Y_s = 2 + true_effect * D_s + 1.5 * U_s + np.random.normal(0, 1, n)

    # First stage F
    stage1_s = smf.ols('D ~ Z', data=pd.DataFrame({'D': D_s, 'Z': Z_s})).fit()
    F = stage1_s.fvalue

    # IV estimate
    cov_yz = np.cov(Y_s, Z_s)[0, 1]
    cov_dz = np.cov(D_s, Z_s)[0, 1]
    iv_est = cov_yz / cov_dz if abs(cov_dz) > 1e-6 else np.nan

    # OLS estimate
    ols_s = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y_s, 'D': D_s})).fit()

    results.append({'gamma': gamma, 'F': F, 'iv': iv_est, 'ols': ols_s.params['D']})

res = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: IV estimate vs instrument strength
ax = axes[0]
ax.plot(res['gamma'], res['iv'], 'o-', color='steelblue', markersize=3, label='IV estimate')
ax.axhline(y=true_effect, color='red', linestyle='--', linewidth=1.5, label=f'True effect = {true_effect}')
ax.axhline(y=res['ols'].mean(), color='gray', linestyle=':', label=f'OLS (biased) ~ {res["ols"].mean():.2f}')
ax.set_xlabel('Instrument strength (gamma)', fontsize=12)
ax.set_ylabel('Estimated treatment effect', fontsize=12)
ax.set_title('IV Estimate vs. Instrument Strength', fontsize=13)
ax.legend()
ax.set_ylim(0, 8)

# Right: F-statistic vs instrument strength
ax = axes[1]
ax.plot(res['gamma'], res['F'], 'o-', color='darkorange', markersize=3)
ax.axhline(y=10, color='red', linestyle='--', linewidth=1.5, label='F = 10 threshold')
ax.set_xlabel('Instrument strength (gamma)', fontsize=12)
ax.set_ylabel('First-stage F-statistic', fontsize=12)
ax.set_title('First-Stage F vs. Instrument Strength', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print("When gamma is small, the instrument barely moves treatment.")
print("The IV estimate is erratic and biased toward OLS.")
print("The F-statistic drops below 10, signaling a weak instrument problem.")

## 4. Weak Instruments

What happens when the instrument barely affects treatment? The first stage is weak, and the IV estimate becomes noisy, biased toward OLS, and unreliable. The standard diagnostic: if the first-stage F-statistic is below 10, worry.

In [ ]:
# === Stage 1: D on Z ===
W1 = np.column_stack([np.ones(n), Z])
beta1 = inv(W1.T @ W1) @ (W1.T @ D_iv)
D_hat = W1 @ beta1

print("=== Stage 1: D = a + b*Z ===")
print(f"  Intercept: {beta1[0]:.3f}")
print(f"  Coef on Z: {beta1[1]:.3f}")

# First-stage F-statistic
stage1 = smf.ols('D ~ Z', data=pd.DataFrame({'D': D_iv, 'Z': Z})).fit()
print(f"  F-statistic: {stage1.fvalue:.1f}  (rule of thumb: F > 10 means strong instrument)")
print()

# === Stage 2: Y on D_hat ===
W2 = np.column_stack([np.ones(n), D_hat])
beta2 = inv(W2.T @ W2) @ (W2.T @ Y_iv)

print("=== Stage 2: Y = a + b*D_hat ===")
print(f"  Intercept:    {beta2[0]:.3f}")
print(f"  Coef on D_hat: {beta2[1]:.3f}")
print()

print(f"True effect:     {true_effect}")
print(f"OLS (biased):    {ols_biased.params['D']:.3f}")
print(f"IV/2SLS:         {beta2[1]:.3f}")
print(f"Wald (Cov ratio):{wald:.3f}")
print(f"\n2SLS and Wald agree (as they should with one instrument and no controls).")

## 3. Two-Stage Least Squares (2SLS) Step by Step

The most common IV estimator. Two stages:

**Stage 1**: Regress $D$ on $Z$. Get predicted values $\hat{D}$. These contain only the "clean" variation in treatment coming from the instrument.

**Stage 2**: Regress $Y$ on $\hat{D}$. The coefficient on $\hat{D}$ is the IV estimate.

Why this works: $\hat{D}$ is a function of $Z$ only, so it is uncorrelated with $U$. By replacing $D$ with $\hat{D}$, we purge the endogenous variation.

In [ ]:
# Instrument: independent of U, affects D, no direct effect on Y
Z = np.random.normal(0, 1, n)

# Treatment now depends on Z AND U
D_iv = 0.6 * Z + 0.5 * U + np.random.normal(0, 1, n)

# Outcome still depends on D and U (Z has NO direct effect)
Y_iv = 2 + true_effect * D_iv + 1.5 * U + np.random.normal(0, 1, n)

# Verify instrument conditions
print("Checking instrument conditions:")
print(f"  Corr(Z, D):  {np.corrcoef(Z, D_iv)[0,1]:.3f}  (should be nonzero: RELEVANCE)")
print(f"  Corr(Z, U):  {np.corrcoef(Z, U)[0,1]:.3f}  (should be ~0: INDEPENDENCE)")
print()

# Wald estimator (for binary Z, but works with continuous Z too via regression)
# IV estimate = Cov(Y, Z) / Cov(D, Z)
cov_YZ = np.cov(Y_iv, Z)[0, 1]
cov_DZ = np.cov(D_iv, Z)[0, 1]
wald = cov_YZ / cov_DZ

# Compare with biased OLS
ols_biased = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y_iv, 'D': D_iv})).fit()

print(f"True effect:    {true_effect}")
print(f"OLS estimate:   {ols_biased.params['D']:.3f}  (biased)")
print(f"IV estimate:    {wald:.3f}  (unbiased!)")
print(f"\nThe IV estimate uses only the variation in D that comes from Z,")
print(f"which is uncorrelated with U. This strips out the confounding.")

## 2. Enter the Instrument

An **instrument** $Z$ satisfies three conditions:

1. **Relevance**: $Z$ is correlated with $D$ (the first stage)
2. **Exclusion**: $Z$ affects $Y$ only through $D$ (no direct effect)
3. **Independence**: $Z$ is uncorrelated with confounders $U$

We add $Z$ to our simulation: $Z$ shifts $D$ but is independent of $U$ and has no direct effect on $Y$.

In [ ]:
n = 5000
true_effect = 3

# Unobserved confounder
U = np.random.normal(0, 1, n)

# Treatment depends on confounder (endogenous!)
D = 0.5 * U + np.random.normal(0, 1, n)

# Outcome depends on treatment AND confounder
Y = 2 + true_effect * D + 1.5 * U + np.random.normal(0, 1, n)

# OLS: regress Y on D (ignoring U because it's unobserved)
ols = smf.ols('Y ~ D', data=pd.DataFrame({'Y': Y, 'D': D})).fit()

print(f"True causal effect of D on Y: {true_effect}")
print(f"OLS estimate:                 {ols.params['D']:.3f}")
print(f"OLS bias:                     {ols.params['D'] - true_effect:+.3f}")
print(f"\nOLS is biased UPWARD because U pushes both D and Y in the same direction.")
print(f"People with high U get more treatment AND have higher outcomes,")
print(f"making treatment look more effective than it really is.")

## 1. The Endogeneity Problem

Suppose we want to estimate the causal effect of treatment $D$ on outcome $Y$. The true model is:

$$Y = 2 + 3D + 1.5U + \varepsilon$$

where $U$ is an unobserved confounder that affects both $D$ and $Y$. Because $D$ depends on $U$, OLS on $Y \sim D$ picks up both the causal effect of $D$ *and* the confounding from $U$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from numpy.linalg import inv

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Instrumental Variables: Concepts and Intuition

When treatment is **endogenous** (correlated with unobserved factors that also affect the outcome), OLS is biased. An **instrumental variable** (IV) is a source of exogenous variation that shifts treatment but has no direct effect on the outcome.

This notebook builds intuition through simulation:
1. Why OLS fails when treatment is endogenous
2. How IV recovers the causal effect
3. Two-stage least squares (2SLS) step by step
4. What goes wrong with weak instruments
5. LATE: who the IV estimate applies to
6. What happens when the exclusion restriction is violated